## Enviar relatório via e-mail Empresa X ↓
Execute para enviar o relatório via email 

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import matplotlib.pyplot as plt
import time
import pyautogui
import xlrd
from openpyxl import Workbook
from openpyxl.styles import Alignment
import openpyxl
from datetime import datetime, timedelta
import pandas as pd
import os
import win32com.client as win32
from io import BytesIO
import base64


# Config data 
data_inicio = input('Insira a data de início (dd/mm/aaaa): ')
data_final = input('Insira a data final (dd/mm/aaaa): ')
Valida_email_teste = input("Digite 1 para teste, ou qualquer botão para enviar aos colaboradores.")



# Config caminhos 
desc_path = r'C:\Users\mauricio.vargas\Documents\Projetos\05 - Empresa X\DescServ.xlsx'


# Busca arquivo
for file_name in os.listdir(r'C:\Users\mauricio.vargas\Downloads'):
    if "RelatorioOrigemDestino" in file_name:
        file_path = os.path.join(r'C:\Users\mauricio.vargas\Downloads', file_name)
        os.remove(file_path)
        print(f"Arquivo apagado: {file_path}")
        
else:
    print("Nenhum arquivo com 'RelatorioOrigemDestino' encontrado na pasta Downloads.")


# -------------------------------------------------------------------------------------------------------------------------------------------
# Acessa Web service
service = Service(ChromeDriverManager().install())
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, timeout=60) 
driver.maximize_window()
url_login = 'https://MinhaUrl.com.br'
driver.get(url_login)
usuario = 'Usuário'
senha = 'MinhaSenha'

#input('Conferir URL HTTP e apertar enter para continuar')


driver.find_element(By.NAME, "j_username").clear()
driver.find_element(By.NAME, "j_username").send_keys(usuario) 

driver.find_element(By.NAME, "j_password").clear()
driver.find_element(By.NAME, "j_password").send_keys(senha)

driver.find_element(By.NAME, "btnAcessar").click()



# -------------------------------------------------------------------------------------------------------------------------------------------
# Config data 
data_inicio_obj = datetime.strptime(data_inicio, "%d/%m/%Y")
data_final_obj = datetime.strptime(data_final, "%d/%m/%Y")

data_relatorio_obj = datetime.strptime(data_inicio, "%d/%m/%Y")
data_relatorio_str = data_relatorio_obj.strftime("%d/%m/%Y")


while data_relatorio_obj <= data_final_obj:
    print('Relatório dia:', data_relatorio_str)
    

    # -------------------------------------------------------------------------------------------------------------------------------------------
    # relatório caminho

    BotaoRelat = wait.until(EC.presence_of_element_located((By.XPATH, '/html/body/div/div/div[3]/div[2]/div[1]/div[1]/div/div/table/tbody/tr/td[9]')))
    BotaoRelat.click()

    time.sleep(1)

    element = driver.find_element(By.PARTIAL_LINK_TEXT, 'Relatórios')
    element.click()

    actions = ActionChains(driver)

    element = driver.find_element(By.PARTIAL_LINK_TEXT, 'Relatórios Estatísticos')
    actions.move_to_element(element).perform()

    element = driver.find_element(By.PARTIAL_LINK_TEXT, 'Desempenho Por Serviço')
    element.click()

    # config relatório ------------------------------------------------------------------------------------------------

    DataIni = wait.until(EC.presence_of_element_located((By.XPATH, '/html/body/div[2]/div[3]/div/div/div/div[1]/div[3]/table/tbody[2]/tr[1]/td[2]/div/i/input')))
    DataIni.click()
    DataIni.send_keys(data_relatorio_str) 
    DataFim = driver.find_element(By.XPATH, '/html/body/div[2]/div[3]/div/div/div/div[1]/div[3]/table/tbody[2]/tr[1]/td[4]/div/i/input')
    DataFim.click()
    DataFim.send_keys(data_relatorio_str) 

    Empresa = driver.find_element(By.XPATH, '/html/body/div[2]/div[3]/div/div/div/div[1]/div[3]/table/tbody[2]/tr[2]/td[2]/div/i/input')
    Empresa.click()
    time.sleep(1)
    Empresa = driver.find_element(By.XPATH, '/html/body/div[3]/table/tbody/tr/td[2]')
    Empresa.click()

    Botao = driver.find_element(By.XPATH, '/html/body/div[2]/div[3]/div/div/div/div[1]/div[3]/table/tbody[2]/tr[4]/td/div/span/input')
    Botao.click()

    Botao = driver.find_element(By.XPATH, '/html/body/div[2]/div[3]/div/div/div/div[1]/div[3]/table/tbody[2]/tr[5]/td/div/span/input')
    Botao.click()

    Botao = driver.find_element(By.XPATH, '/html/body/div[2]/div[3]/div/div/div/div[2]/div[1]/button')
    Botao.click()

    BotaoExcel = wait.until(EC.presence_of_element_located((By.XPATH, '/html/body/div[4]/div[3]/div/div/div/div/div[1]/div[1]/div[2]')))
    BotaoExcel.click()
    
    #pyautogui.click(1198,109,duration=3)

    time.sleep(3)
 
    #input('aperte enter para continuar')
    
    # busca arquivo ------------------------------------------------------------------------------------------------
    for file_name in os.listdir(r'C:\Users\mauricio.vargas\Downloads'):
        if "RelatorioOrigemDestino" in file_name:
            file_path = os.path.join(r'C:\Users\mauricio.vargas\Downloads', file_name)
            print(f"Arquivo encontrado: {file_path}")
            break
    else:
        print("Nenhum arquivo com 'RelatorioOrigemDestino' encontrado na pasta Downloads.")
        
    # -------------------------------------------------------------------------------------------------------------------------------------------



    # Abre o arquivo .xls com xlrd
    workbook = xlrd.open_workbook(file_path)
    sheet = workbook.sheet_by_name('RelatorioOrigemDestino')

    # Cria um novo arquivo Excel .xlsx para armazenar o filtro
    wb = Workbook()
    ws = wb.active
    ws.title = "Filtrado"

    # Copia o cabeçalho
    for col in range(sheet.ncols):
        ws.cell(row=1, column=col + 1, value=sheet.cell_value(0, col))

    # Filtra os dados (assumindo que 'Empresa' está na coluna 0) e remove linhas em branco
    row_index = 2
    for row in range(1, sheet.nrows):
        empresa = sheet.cell_value(row, 0)
        if empresa == 'Empresa X' or empresa == '':
            # Insere a linha completa, mesmo se houver células vazias
            for col in range(sheet.ncols):
                ws.cell(row=row_index, column=col + 1, value=sheet.cell_value(row, col))
            row_index += 1

    # Alinhar os dados centralizados
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
        for cell in row:
            cell.alignment = Alignment(horizontal="center", vertical="center")

    # Escreve "PASSAGEIROS" na célula G1 e "FATURAMENTO" na célula H1
    ws['G1'] = 'Passageiros'
    ws['H1'] = 'Faturamento'

    # Deleta as células de G2 a M2 e desloca os dados para cima
    for col in range(7, 14):
        for row in range(2, ws.max_row):
            ws.cell(row=row, column=col).value = ws.cell(row=row + 1, column=col).value

    # Limpa os valores duplicados na última linha
    for col in range(7, 14):
        ws.cell(row=ws.max_row, column=col).value = None

    # Deletar colunas específicas
    ws.delete_cols(1)  
    ws.delete_cols(2) 
    ws.delete_cols(2) 
    ws.delete_cols(2) 
    ws.delete_cols(2)  

    # Apaga os valores das colunas D até H
    for col in range(4, 9):
        for row in range(1, ws.max_row + 1):
            ws.cell(row=row, column=col).value = None

    # Aplicar filtro na primeira linha
    ws.auto_filter.ref = "A1:C24"

    # Ordenar a coluna "Serviço"
    data_rows = list(ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=3, values_only=True))
    sorted_rows = sorted([row for row in data_rows if row[0] is not None], key=lambda row: str(row[0]))

    # Limpa as linhas e reescreve com dados ordenados
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=3):
        for cell in row:
            cell.value = None

    for idx, row_data in enumerate(sorted_rows, start=2):
        for col_idx, value in enumerate(row_data, start=1):
            ws.cell(row=idx, column=col_idx, value=value)

    # Formatar "Faturamento" como valor monetário BRL
    for row in range(2, ws.max_row + 1):
        cell = ws.cell(row=row, column=3)
        if isinstance(cell.value, (int, float)):
            cell.number_format = '"R$"#,##0.00'

    data = []
    columns = [cell.value for cell in ws[1]] 

    # Percorrer as linhas restantes para coletar os dados
    for row in ws.iter_rows(min_row=2, values_only=True):
        if any(row): 
            data.append(dict(zip(columns, row)))

    # Transformar os dados extraídos em um DataFrame
    fin_df = pd.DataFrame(data)
    fin_df = fin_df.dropna(axis=1, how='all')

    # mescla arquivos ------------------------------------------------------------------------------------------------
    desc_df = pd.read_excel(desc_path)
    desc_guia2_df = pd.read_excel(desc_path, sheet_name="Linhas")
    merged_df = pd.merge(fin_df, desc_df, on="Serviço", how="left")

    # ----------------------------------------------------------------------------------------------------------------
    # Preenche a coluna "Data" apenas para as linhas que têm dados nas colunas principais

    mask = merged_df[['Serviço', 'Passageiros', 'Faturamento']].notnull().all(axis=1)
    merged_df.loc[mask, 'Data'] = data_relatorio_str

    df_final = merged_df.dropna(subset=['Serviço', 'Passageiros', 'Faturamento'], how='any')


    df_final = df_final.copy()
    df_final['RPK'] = (df_final['Faturamento'] / df_final['KM Total da linha']).round(2)


    # Mover a nova coluna 'RPK' para logo após a coluna 'Meta' 
    cols = df_final.columns.tolist()
    meta_index = cols.index('Meta')
    cols.insert(meta_index + 1, cols.pop(cols.index('RPK')))
    df_final = df_final[cols]
    os.remove(file_path)

    # Agrupar os dados por 'Linha' e somar 'Faturamento' e 'KM Total da linha'
    df_resumo = df_final.groupby('Linha').agg({
        'Faturamento': 'sum',
        'KM Total da linha': 'sum',
    }).reset_index()

    # Adicionar a coluna 'Viagens', que conta o número de vezes que cada linha apareceu
    df_resumo['Viagens'] = df_final.groupby('Linha').size().values

    # Calcular o RPK como Faturamento Total / KM Total da Linha
    df_resumo['RPK'] = df_resumo['Faturamento'] / df_resumo['KM Total da linha']

    # Convertendo a coluna 'Linha' para string para garantir que cada linha seja tratada individualmente
    df_resumo['Linha'] = df_resumo['Linha'].astype(str)

    

    # -------------------------------------------------------------------------------------------------------------------------------------------
    # Salvando relatório em donwloads


    dia_relatorio = data_relatorio_str[:2]
    data_final_edit = data_final.replace('/', '-')
    df_final.to_excel(f"~/Downloads/RelServ {dia_relatorio} a {data_final_edit}.xlsx", index=False)
    print('Relatório salvo com sucesso na pasta "Downloads"!')

    # soma +1 na data do relatório
    data_relatorio_obj += timedelta(days=1)
    data_relatorio_str = data_relatorio_obj.strftime("%d/%m/%Y")
    driver.get(url_login)


# -------------------------------------------------------------------------------------------------------------------------------------------
# Concatena arquivos gerados na pasta downloads

diretorio_downloads = "C:/Users/mauricio.vargas/Downloads"

# Lista para armazenar cada DataFrame carregado
df_RelServ = []

# Percorrer todos os arquivos no diretório
for filename in os.listdir(diretorio_downloads):
    if "RelServ" in filename and filename.endswith(".xlsx"):
        file_path = os.path.join(diretorio_downloads, filename)
        df = pd.read_excel(file_path)  
        df_RelServ.append(df)  

# Concatenar todos os DataFrames em um único
df_RelServ_final = pd.concat(df_RelServ, ignore_index=True)

# Salvar o DataFrame final em um novo arquivo Excel
df_RelServ_final.to_excel("~/Downloads/Rel_concatenado.xlsx", index=False)



driver.quit()

# limpa todos os arquivos temporários
for filename in os.listdir(diretorio_downloads):
    if "RelServ" in filename and filename.endswith(".xlsx"):
        file_path  = os.path.join(diretorio_downloads, filename)
        os.remove(file_path)  # Usar os.remove() para deletar o arquivo
        print(f"Arquivo deletado: {file_path}")


df_RelServ_final = pd.read_excel('~/Downloads/Rel_concatenado.xlsx')
df_final = df_RelServ_final

# Apaga arquivo temporário
os.remove('C:/Users/mauricio.vargas/Downloads/Rel_concatenado.xlsx')


# Agrupar os dados por 'Linha' e somar 'Faturamento' e 'KM Total da linha'
df_resumo = df_final.groupby('Linha').agg({
    'Faturamento': 'sum',
    'KM Total da linha': 'sum',
}).reset_index()

# Adicionar a coluna 'Viagens', que conta o número de vezes que cada linha apareceu
df_resumo['Viagens'] = df_final.groupby('Linha').size().values

# Calcular o RPK como Faturamento Total / KM Total da Linha
df_resumo['RPK'] = df_resumo['Faturamento'] / df_resumo['KM Total da linha']

# Convertendo a coluna 'Linha' para string para garantir que cada linha seja tratada individualmente
df_resumo['Linha'] = df_resumo['Linha'].astype(str)


# -------------------------------------------------------------------------------------------------------------------------------------------
# Função para adicionar os valores no topo das barras do gráfico

def add_value_labels(ax, spacing=0, format_str="R${:,.2f}", fontsize=8):
    """ Adiciona rótulos de valor no topo das barras com espaçamento reduzido """
    for rect in ax.patches:
        height = rect.get_height()
        ax.text(
            rect.get_x() + rect.get_width() / 2,  
            height + spacing,  # Posição vertical (topo da barra)
            format_str.format(height),  
            ha='center', va='bottom', fontsize=fontsize  
        )

# -------------------------------------------------------------------------------------------------------------------------------------------
# Criando gráfico de barras para Faturamento por Linha

plt.figure(figsize=(15, 6))
ax = plt.bar(df_resumo['Linha'], df_resumo['Faturamento'], color='tab:blue', width=0.8)  # Ajustando o espaçamento entre as barras
plt.xlabel('Linha')
plt.ylabel('Faturamento')
plt.title('Faturamento por Linha')
plt.xticks(rotation=90) 
plt.tight_layout()

# Adicionando os rótulos no topo das barras com tamanho menor e menor espaçamento
ax = plt.gca() 
add_value_labels(ax, format_str="R${:,.2f}", fontsize=8, spacing=1) 

# Salvar o gráfico em um buffer (Faturamento)
buffer_faturamento = BytesIO()
plt.savefig(buffer_faturamento, format='png')
plt.close()

# Converter o gráfico de Faturamento para string base64
buffer_faturamento.seek(0)
img_faturamento_base64 = base64.b64encode(buffer_faturamento.read()).decode('utf-8')

# Criando gráfico de barras para RPK por Linha
plt.figure(figsize=(15, 6))
ax = plt.bar(df_resumo['Linha'], df_resumo['RPK'], color='tab:orange', width=0.8) 
plt.xlabel('Linha')
plt.ylabel('RPK')
plt.title('RPK por Linha')
plt.xticks(rotation=90) 

# Adicionando a linha horizontal na faixa de 8.00
plt.axhline(y=8.00, color='red', linestyle='--', linewidth=2, label='Faixa de R$8.00')

# Adicionando os rótulos no topo das barras com tamanho menor e menor espaçamento
ax = plt.gca()
add_value_labels(ax, format_str="R${:,.2f}", fontsize=8, spacing=0) 
plt.tight_layout()
plt.legend()

# Salvar o gráfico em um buffer (RPK)
buffer_rpk = BytesIO()
plt.savefig(buffer_rpk, format='png')
plt.close()

# Converter o gráfico de RPK para string base64
buffer_rpk.seek(0)
img_rpk_base64 = base64.b64encode(buffer_rpk.read()).decode('utf-8')
# -------------------------------------------------------------------------------------------------------------------------------------------
# Valor total
VTOT_periodo = f"{df_RelServ_final['Faturamento'].sum():,.2f}".replace('.', '-').replace(',', '.').replace('-', ',')
KMTOT_periodo = f"{df_RelServ_final['KM Total da linha'].sum():,.2f}".replace('.', '-').replace(',', '.').replace('-', ',')
RPKTOT_periodo = df_RelServ_final['Faturamento'].sum() / df_RelServ_final['KM Total da linha'].sum()
RPKTOT_periodo = f"{RPKTOT_periodo:,.2f}".replace('.', '-').replace(',', '.').replace('-', ',')
PASSTOT = df_RelServ_final['Passageiros'].sum()

# Converte em string a aba linhas e insere descrição
df_resumo['Linha'] = df_resumo['Linha'].astype(str).str.strip()
desc_guia2_df['Linha'] = desc_guia2_df['Linha'].astype(str).str.strip()

# Faz o merge com descrição das linhas
df_resumo = df_resumo.merge(desc_guia2_df[['Linha', 'Descrição']], on='Linha', how='left')

# Ordena do maior para o menor pelo KM
df_resumo = df_resumo.sort_values(by="KM Total da linha", ascending=False)

# Formatando DF resumo
df_resumo['Faturamento'] = df_resumo['Faturamento'].map(lambda x: f"R${x:,.2f}".replace('.', 'X').replace(',', '.').replace('X', ','))
df_resumo['RPK'] = df_resumo['RPK'].map(lambda x: f"R${x:,.2f}".replace('.', 'X').replace(',', '.').replace('X', ','))
df_resumo['KM Total da linha'] = df_resumo['KM Total da linha'].map(lambda x: f"{x:,.2f} KM".replace('.', 'X').replace(',', '.').replace('X', ','))

# Identifica serviços não cadastrados
dados_vazios = df_final[df_final.isnull().any(axis=1)]
if not dados_vazios.empty:
    print("Linhas com serviços não cadastrados:")
    display(dados_vazios)

# Formatando DF final
df_final['Faturamento'] = df_final['Faturamento'].map(lambda x: f"R${x:,.2f}".replace('.', 'X').replace(',', '.').replace('X', ','))
df_final['RPK'] = df_final['RPK'].map(lambda x: f"R${x:,.2f}".replace('.', 'X').replace(',', '.').replace('X', ','))
df_final['KM Total da linha'] = df_final['KM Total da linha'].map(lambda x: f"{x:,.2f} KM".replace('.', 'X').replace(',', '.').replace('X', ','))
df_final['Serviço'] = df_final['Serviço'].astype(int)
df_final['Passageiros'] = df_final['Passageiros'].astype(int)
df_final['Meta'] = df_final['Meta'].astype(int)

df_final['Linha'] = df_final['Linha'].astype(str)
df_final = df_final.sort_values(by='Linha', ascending=True)


# -------------------------------------------------------------------------------------------------------------------------------------------
# EMAIL

# Aplicar a estilização no DataFrame final
df_final = df_final.style.applymap(
    lambda x: "background-color: purple; color: white;" if x == "Modificado" else
              "background-color: yellow; color: black;" if x == "Aproveitamento" else
              "background-color: lightblue; color: black;" if x == "Extra" else
              "background-color: #FFC000; color: white;" if x == "Temporário" else
            "",
    subset=["Situação"]
)

# Convertendo o DataFrame estilizado para HTML sem índice
df_html_final = df_final.hide(axis="index").to_html()

# Convertendo o DataFrame resumo para HTML sem índice
df_html_resumo = df_resumo.to_html(index=False).replace('<td>', '<td style="text-decoration: none;">').replace('<th>', '<th style="text-decoration: none;">')


# Destacar as células que contêm "1186" com fundo amarelo
df_html_final = df_html_final.replace(
    '>1186 B<', ' style="background-color: yellow;">1186 B<'
).replace(
    '>1009<', ' style="background-color: yellow;">1009<'
)

df_html_resumo = df_html_resumo.replace(
    '>1186 B<', ' style="background-color: yellow;">1186 B<'
).replace(
    '>1009<', ' style="background-color: yellow;">1009<'
)


# Construir o conteúdo do e-mail com o DataFrame e o gráfico
conteudo_email = f"""
<html>
<head>
    <style>
        .valor {{
            color: green;
            font-weight: bold;
            font-size: 1.2em; 
        }}
        table {{
            font-family: Arial, sans-serif;
            border-collapse: collapse;
            width: 100%;
        }}
        table, th, td {{
            border: 1px solid black;
            padding: 8px;
        }}
    </style>
</head>
<body>
    <p>Relatório parcial de vendas EMPRESA X de {data_inicio} a {data_final} </p>
    
    <h1>Resumo de vendas Geral - <span style="color: #C00000;">Empresa X</span></h1>

    O total faturado é: <span class="valor">R${VTOT_periodo}▲</span><br>

    Km total percorrido: <span class="valor">{KMTOT_periodo}Km</span><br>

    Receita por km: <span class="valor">R${RPKTOT_periodo}▲</span><br>

    O total de bilhetes vendidos é: <span class="valor">{PASSTOT}</span><br>
    

    <br>

    <h1>Resumo de vendas por linhas</h1>
    {df_html_resumo}  <!-- Inclui o DataFrame em formato HTML -->
    
    <br>
    
    <img src="data:image/png;base64,{img_faturamento_base64}" alt="Gráfico de Vendas" />
    <img src="data:image/png;base64,{img_rpk_base64}" alt="Gráfico de Vendas" />
    <br>
    <br>
    
    <h1>Resumo de vendas por horários</h1>
    
    {df_html_final}
    
    <br>
    <p>Atenciosamente,</p>
    <p>Mauricio Vargas</p>
</body>
</html>
"""


# Configuração do e-mail
if Valida_email_teste == '1':
    destinatario = "TESTE@teste1.com.br;"
else:
    destinatario = "Diretoria@empresa.com.br;gerencia@empresa.com.br;"



# Defina o e-mail do remetente desejado
email_envio = "MeuEmail@empresa.com.br"

assunto = f"Relatório de Vendas Empresa X - {data_inicio} À {data_final}"
corpo = conteudo_email

# Iniciar o Outlook
outlook = win32.Dispatch('Outlook.Application')

# Criar o e-mail
email = outlook.CreateItem(0)
email.To = destinatario
email.Subject = assunto
email.HTMLBody = corpo

# Selecionar a conta correta para envio
for account in outlook.Session.Accounts:
    if account.SmtpAddress.lower() == email_envio.lower():
        email._oleobj_.Invoke(*(64209, 0, 8, 0, account))  # Força a conta de envio correta
        break
else:
    raise ValueError(f"Conta de e-mail '{email_envio}' não encontrada no Outlook.")

# Enviar o e-mail
email.Send()
print("---------------------------------------------------------------------")
print(f"E-mail enviado com sucesso a partir de: {email_envio}")
# -------------------------------------------------------------------------------------------------------------------------------------------




## Enviar relatório Empresa Y 2.0 ↓
Execute para enviar o relatório via email

In [ ]:
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium import webdriver
import win32com.client as win32
from datetime import datetime, timedelta
import pandas as pd
import time
import os


DATA_INICIO = input("insira a data de inicio:")
DATA_FIM = input("insira a data de fim:")
Valida_email_teste = input("Digite 1 para teste, ou qualquer botão para enviar aos colaboradores.")

# Caminho manual para o ChromeDriver
base_dir = os.path.expanduser(r'~\.wdm\drivers\chromedriver\win64\127.0.6533.72\chromedriver-win32')
chrome_driver_path = os.path.join(base_dir, 'chromedriver.exe')
service = Service(ChromeDriverManager().install())
options = webdriver.ChromeOptions()

# Inicializando o driver
driver = webdriver.Chrome(service=service, options=options)

# Abre o site de login
url_login = 'https://URL.com.br' 
driver.get(url_login)
driver.maximize_window()

# Defina as informações de login
usuario = 'User'
senha = 'senha'

# Encontra o campo de usuário e insere o nome de usuário
campo_usuario = driver.find_element(By.ID, "usuario") 
campo_usuario.clear()
campo_usuario.send_keys(usuario)
campo_senha = driver.find_element(By.ID, "senha")
campo_senha.clear()
campo_senha.send_keys(senha)

botao_login = driver.find_element(By.ID, "entrar")
botao_login.click()

time.sleep(10)

dropdown = driver.find_element(By.XPATH, "/html/body/aside/nav/div/ul/li[5]")
dropdown.click()

dropdown = driver.find_element(By.XPATH, "//*[@id='menu-relatorioFinanceiro']/div[4]/div/div/li[18]")
dropdown.click()

# Insere data e parâmetros
# -------------------------------------------------------------------------------------------------------------------------------------------
# cria df vazio
df_final = pd.DataFrame()

# Config data 
data_inicio_obj = datetime.strptime(DATA_INICIO, "%d/%m/%Y")
data_final_obj = datetime.strptime(DATA_FIM, "%d/%m/%Y")

data_relatorio_obj = datetime.strptime(DATA_INICIO, "%d/%m/%Y")
data_relatorio_str = data_relatorio_obj.strftime("%d/%m/%Y")


while data_relatorio_obj <= data_final_obj:
    print('Relatório dia:', data_relatorio_str)

    Botao_DataIni = driver.find_element(By.ID, "RelVendaPass-DateTime-DataIni")
    Botao_DataIni.clear()
    Botao_DataIni.send_keys(data_relatorio_str)

    Botao_DataFim = driver.find_element(By.ID, "RelVendaPass-DateTime-DataFim")
    Botao_DataFim.clear()
    Botao_DataFim.send_keys(data_relatorio_str)

    body = driver.find_element(By.TAG_NAME, 'body')
    body.click()

    Botao_Partida = driver.find_element(By.XPATH, '/html/body/div/div[1]/div[1]/div/div/form/fieldset/div/div[1]/div[3]/fieldset/div/div[2]/div/label')
    Botao_Partida.click()

    Botao_Servico = driver.find_element(By.XPATH, '//*[@id="form-RelVendaPassagem"]/fieldset/div/div[6]/div[2]/div/label')
    driver.execute_script("arguments[0].scrollIntoView();", Botao_Servico)
    
    Botao_Cancelados = driver.find_element(By.XPATH, '/html/body/div/div[1]/div[1]/div/div/form/fieldset/div/div[7]/div[3]/div/label')
    
    time.sleep(0.5)

    if data_relatorio_str == DATA_INICIO:
        Botao_Servico.click()
        Botao_Cancelados.click()
    else:
        pass

        

        

    Botao_Extrair = driver.find_element(By.XPATH, '/html/body/div/div[1]/div[1]/div/div/form/div[2]/div/div/button[2]')
    Botao_Extrair.click()

    time.sleep(2)
    Botao_Excel = driver.find_element(By.XPATH, '/html/body/div/div[1]/div[1]/div/div/form/div[2]/div/div/div/a[5]')
    Botao_Excel.click()

    diretorio = os.path.join(os.path.expanduser("~"), "Downloads")
    caminho_relatorio_xlsx = None
    while caminho_relatorio_xlsx is None:
        for caminho_pasta, subpastas, arquivos in os.walk(diretorio):
            for nome_arquivo in arquivos:
                if "RELATÓRIO VENDA PASSAGEM" in nome_arquivo.upper():
                    caminho_relatorio_xlsx = os.path.join(caminho_pasta, nome_arquivo)
                    print('Relatório encontrado:', caminho_relatorio_xlsx)
                    break
            if caminho_relatorio_xlsx is not None:
                break
        else:
            print("Relatório Venda Passagem não encontrado. Tentando novamente em 5 segundos...")
            time.sleep(5)
        

    # Importa excel
    df = pd.read_excel(caminho_relatorio_xlsx)

    # Exclui excel
    os.remove(caminho_relatorio_xlsx)

    data_relatório = data_relatorio_str
    data_relatório_objeto = datetime.strptime(data_relatório, "%d/%m/%Y")

    # Limpa dados
    df = df[['Partida', 'Total', 'Tipo Passageiro - Forma Pagamento', 'Serviço', 'Tipo Passageiro']]
    df_remove = df.loc[(df['Tipo Passageiro'] != 'Comum') | (df['Partida'] != data_relatório)]
    df = df.drop(df_remove.index)
    df = df.drop('Tipo Passageiro', axis=1)

    # Contar a quantidade de vezes que cada número de "Serviço" aparece
    df['Passageiros'] = df.groupby('Serviço')['Serviço'].transform('count')

    # Agrupar pela coluna 'Serviço' e somar a coluna 'Total', mantendo a contagem de 'Passageiros'
    soma_servico = df.groupby('Serviço').agg({'Total': 'sum', 'Passageiros': 'first'})
    df_servico = soma_servico.reset_index()
    df_servico = df_servico.rename(columns={'Total': 'Faturamento'})
    df_servico['Data'] = df['Partida']

    # Agrupa com a coluna descrição
    df_DescServ = pd.read_excel('C:/Users/mauricio.vargas/Documents/Projetos/05 - EmpresaX 2/EmpresaX - DescServ.xlsx')
    df_prov = pd.merge(df_servico, df_DescServ, on='Serviço')
    df_prov['RPK'] = df_prov['Faturamento'] / df_prov['KM Total da linha']

    #Arredondando RPK e organizando dados
    df_prov['RPK'] = df_prov['RPK'].round(2)
    df_prov['Linha'] = df_prov['Linha'].astype(str)
    df_prov = df_prov.sort_values(by='Linha')

    df_final = pd.concat([df_final, df_prov], ignore_index=True)
    # soma +1 na data do relatório
    data_relatorio_obj += timedelta(days=1)
    data_relatorio_str = data_relatorio_obj.strftime("%d/%m/%Y")
    df_final = df_final.sort_values(by="Data", ascending=True)
    #driver.get(url_login)



# Cálcula totais
Fat_Total = df_final['Faturamento'].sum()


#Km total sem contabilizar a linha 1080 A
Km_Total = df_final[df_final["Linha"] != "1080 A"]["KM Total da linha"].sum()


Passageiros = df_final['Passageiros'].sum()

# Cálcula RPK
RPK = Fat_Total / Km_Total


# Criando DF resumo das duas linhas
df_resumo = df_final.groupby('Linha').agg({
        'Faturamento': 'sum',
        'KM Total da linha': 'sum',
    }).reset_index()
    


driver.quit()
#del df, df_servico, df_DescServ


# Ajustando km da 1080 A (aproveitamento) ---------------------------------------------------------------------------------------
df_resumo.loc[df_resumo['Linha'] == '1080 A', 'KM Total da linha'] = 1 
# Adicionar a coluna 'Viagens', que conta o número de vezes que cada linha apareceu
df_resumo['Viagens'] = df_final.groupby('Linha').size().values
df_resumo['RPK'] = df_resumo['Faturamento'] / df_resumo['KM Total da linha']
df_resumo['Linha'] = df_resumo['Linha'].astype(str)

# Formata Variável em padrão BRL String 
Km_Total_String = f'{Km_Total:,.2f}'.replace('.', '*').replace(',', '.').replace('*', ',')
Fat_Total_String = f'{Fat_Total:,.2f}'.replace('.', '*').replace(',', '.').replace('*', ',')
RPK_String = f'{RPK:,.2f}'.replace('.', '*').replace(',', '.').replace('*', ',')

# Formata em padrão BRL e formata em string
df_resumo['Faturamento'] = df_resumo['Faturamento'].map(lambda x: f"R${x:,.2f}".replace('.', 'X').replace(',', '.').replace('X', ','))
df_resumo['RPK'] = df_resumo['RPK'].map(lambda x: f"R${x:,.2f}".replace('.', 'X').replace(',', '.').replace('X', ','))
df_resumo['KM Total da linha'] = df_resumo['KM Total da linha'].map(lambda x: f"{x:,.2f} KM".replace('.', 'X').replace(',', '.').replace('X', ','))

# Formata em padrão BRL e formata em string
df_final['Faturamento'] = df_final['Faturamento'].map(lambda x: f"R${x:,.2f}".replace('.', 'X').replace(',', '.').replace('X', ','))
df_final['RPK'] = df_final['RPK'].map(lambda x: f"R${x:,.2f}".replace('.', 'X').replace(',', '.').replace('X', ','))
df_final['KM Total da linha'] = df_final['KM Total da linha'].map(lambda x: f"{x:,.2f} KM".replace('.', 'X').replace(',', '.').replace('X', ','))
df_final['Serviço'] = df_final['Serviço'].astype(int)
df_final['Passageiros'] = df_final['Passageiros'].astype(int)
df_final['Meta'] = df_final['Meta'].astype(int)


# Aplicar a estilização no DataFrame final
df_final_styled = df_final.style.applymap(
    lambda x: "background-color: purple; color: white;" if x == "Modificado" else
              "background-color: yellow; color: black;" if x == "Aproveitamento" else
              "background-color: lightblue; color: black;" if x == "Extra" else
              "",
    subset=["Situação"]
)

# Convertendo o DataFrame estilizado para HTML
df_final_html = df_final_styled.hide(axis="index").to_html()

# Convertendo o DataFrame resumo para HTML
df_resumo_html = df_resumo.to_html(index=False).replace('<td>', '<td style="text-decoration: none;">').replace('<th>', '<th style="text-decoration: none;">')

# --------------------------------------------------------------------------------------
# Conteúdo do e-mail em HTML
conteudo_email = f"""
<html>
<head>
    <style>
        .valor {{
            color: green;
        }}
        table {{
            font-family: Arial, sans-serif;
            border-collapse: collapse;
            width: 100%;
        }}
        table, th, td {{
            border: 1px solid black;
            padding: 8px;
        }}
        .valor {{
            color: green;
            font-weight: bold;
            font-size: 1.2em; 
        }}

    </style>
</head>
<body>


    <p>Relatório parcial de vendas Empresa Y de {DATA_INICIO} À {DATA_FIM}</p>

    <h1>Resumo de vendas Geral - <span style="color: #f0cd43;">Empresa Y</span></h1>
    
    O total faturado é: <span class="valor">R${Fat_Total_String}▲</span><br>

    Km total percorrido: <span class="valor">{Km_Total_String}Km</span><br>

    Receita por km: <span class="valor">R${RPK_String}▲</span><br>

    O total de bilhetes vendidos é: <span class="valor">{Passageiros}</span><br>
    

    <br><br><br>
    <h1>Resumo de vendas por linhas</h1>

    {df_resumo_html}  <!-- Inclui o DataFrame em formato HTML -->

    <br>

    <h1>Resumo de vendas por horários</h1>

    {df_final_html}

    <br>

    <p>Atenciosamente,</p>
    <p>Mauricio Vargas</p>
</body>
</html>
"""



# Configuração do e-mail
if Valida_email_teste == '1':
    destinatario = "TESTE@teste1.com.br;"

elif Valida_email_teste == '2':
    destinatario = "qualidade@EmpresaY.com.br;testes@EmpresaY.com;"
    
else:
        destinatario = """diretor@EmpresaY.com.br;
            gerencia@EmpresaY.com.br;
            administrativo@EmpresaY.com.br;
            assessoria@EmpresaY.com.br;
            comercial@EmpresaY.com.br;
            qualidade@EmpresaY.com.br
            """


# Defina o e-mail do remetente desejado
email_envio = "meuemail@EmpresaY.com.br"

assunto = f"Relatório de Vendas EmpresaY - {DATA_INICIO} À {DATA_FIM}"
corpo = conteudo_email

# Iniciar o Outlook
outlook = win32.Dispatch('Outlook.Application')

# Criar o e-mail
email = outlook.CreateItem(0)
email.To = destinatario
email.Subject = assunto
email.HTMLBody = corpo

# Selecionar a conta correta para envio
for account in outlook.Session.Accounts:
    if account.SmtpAddress.lower() == email_envio.lower():
        email._oleobj_.Invoke(*(64209, 0, 8, 0, account))  # Força a conta de envio correta
        break
else:
    raise ValueError(f"Conta de e-mail '{email_envio}' não encontrada no Outlook.")

# Enviar o e-mail
email.Send()
print(f"E-mail enviado com sucesso a partir de: {email_envio}")
# -------------------------------------------------------------------------------------------------------------------------------------------







## Enviar relatório EMPRESA Z MONITORAMENTO ↓
Execute para enviar o relatório via email

In [ ]:
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import win32com.client as win32
from selenium import webdriver
import matplotlib.pyplot as plt
from datetime import datetime
from io import BytesIO
import pandas as pd
import pyautogui
import base64
import time
import os

import plotly.express as px


# Config data 
data_inicio = input('Insira a data de início (dd/mm/aaaa): ')
data_final = input('Insira a data final (dd/mm/aaaa): ')
Valida_email_teste = input("Digite 1 para teste, 2 para nao enviar, ou qualquer botão para enviar aos colaboradores.")
Apg_arq_temp = input("Aperte 1 para manter arquivos temporários.")

# Apaga arquivos sitrack.csv
for file_name in os.listdir(r'C:\Users\mauricio.vargas\Downloads'):
    if "sitrack" in file_name:
        file_path = os.path.join(r'C:\Users\mauricio.vargas\Downloads', file_name)
        os.remove(file_path)
        print(f"Arquivo apagado: {file_path}")
        
else:
    #print("Nenhum arquivo com 'sitrack.csv' encontrado na pasta Downloads.")
    pass

# Caminho Download
Path_download = r'C:\Users\mauricio.vargas\Downloads'

# Inicializa driver com ChromeDriverManager
options = webdriver.ChromeOptions()
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
driver.maximize_window()


def ReNomeArquivo(Nome_Arquivo):

    while True:
        encontrado = False
        for file_name in os.listdir(Path_download):
            if "sitrackPR" in file_name:
                encontrado = True
                old_path = os.path.join(Path_download, file_name)
                Nome_Arquivo = f'sitrack({Nome_Arquivo}).csv'
                new_path = os.path.join(Path_download, Nome_Arquivo)

                os.rename(old_path, new_path)
                print(f"Arquivo renomeado: {Nome_Arquivo}")
                return  # encerra a função após sucesso

        if not encontrado:
            print("Aguardando arquivo com 'sitrackPR' na pasta Downloads...")
            time.sleep(2)

def Loga_extrai (usuario_AA, senha_AA, Nome_Arquivo):
    url_login = 'https://URL.MONITORAMENTO.COM.BR'
    driver.get(url_login)


    driver.find_element(By.NAME, "userName").clear()
    driver.find_element(By.NAME, "userName").send_keys(usuario_AA) 

    driver.find_element(By.NAME, "password").clear()
    driver.find_element(By.NAME, "password").send_keys(senha_AA) 

    driver.find_element(By.ID, "loginButton").click()


    driver.find_element(By.CSS_SELECTOR, "[title='Consultas']").click()

    time.sleep(0.5)

    driver.find_element(By.CSS_SELECTOR, "[title='Históricos']").click()

    time.sleep(0.5)

    driver.find_element(By.CSS_SELECTOR, "[title='H. Posição']").click()

    # - acessa o IFRAME
    iframe = driver.find_element(By.NAME, "tab3")  # Substitua "tab3" pelo atributo adequado do iframe, se necessário
    driver.switch_to.frame(iframe)

    time.sleep(0.5)

    driver.find_element(By.NAME, "rawdatefield-1037-inputEl").clear()
    driver.find_element(By.NAME, "rawdatefield-1037-inputEl").send_keys(data_inicio) 

    driver.find_element(By.NAME, "rawdatefield-1039-inputEl").clear()
    driver.find_element(By.NAME, "rawdatefield-1039-inputEl").send_keys(data_final) 

    driver.find_element(By.ID, "splitbutton-1041").click()
    # ----------------------------------------------------------------------------------------------------------------
    
    Tabela_Evento = None
    trigger = None
    action = None

    # Localiza o título da coluna com texto "Evento"
    Tabela_Evento = WebDriverWait(driver, 3).until(
        EC.visibility_of_element_located((By.XPATH, "//span[text()='Evento']/ancestor::div[contains(@class, 'x-column-header')]"))
    )
    ActionChains(driver).move_to_element(Tabela_Evento).perform()

    # Aguarda a seta (trigger) que aparece no hover
    trigger = WebDriverWait(driver, 3).until(
        EC.element_to_be_clickable((By.XPATH, "//span[text()='Evento']/ancestor::div[contains(@class, 'x-column-header')]//div[contains(@id, 'triggerEl')]"))
    )
    driver.execute_script("arguments[0].scrollIntoView(true);", trigger)
    trigger.click()

    # ----------------------------------------------------------------------------------------------------------------
    time.sleep(3)
    
    WebDriverWait(driver, 3).until(EC.element_to_be_clickable((By.XPATH, "//span[text()='Colunas']"))).click()

    # motorista 
    # Espera o elemento aparecer
    elemento = WebDriverWait(driver, 3).until(
        EC.presence_of_element_located((By.XPATH, "/html/body/div[7]/div/div[2]/div[2]/div[21]"))
    )

    # Rola até ele ficar visível
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", elemento)

    # Espera estar clicável e clica - MOTORISTA
    WebDriverWait(driver, 3).until(
        EC.element_to_be_clickable((By.XPATH, "/html/body/div[7]/div/div[2]/div[2]/div[21]"))
    ).click()

    # LAT E LONG
    WebDriverWait(driver, 3).until(
        EC.element_to_be_clickable((By.XPATH, "/html/body/div[7]/div/div[2]/div[2]/div[16]"))
    ).click()
    WebDriverWait(driver, 3).until(
        EC.element_to_be_clickable((By.XPATH, "/html/body/div[7]/div/div[2]/div[2]/div[17]"))
    ).click()

    #input('aaaaa')

    # ----------------------------------------------------------------------------------------------------------------
    WebDriverWait(driver, 3).until(EC.element_to_be_clickable((By.XPATH, "//span[text()='Condições']"))).click()   
    
    time.sleep(1)

    pyautogui.press('tab')
    pyautogui.press('tab')

    pyautogui.sleep(2)

    # Escreve o texto
    pyautogui.write('em velocidade')
    time.sleep(2)   
    pyautogui.press('down')
    time.sleep(1)
    pyautogui.press('enter')
    pyautogui.press('tab')
    pyautogui.press('tab')
    pyautogui.press('enter')
    # --------------------------------------------------------------------

    time.sleep(3)

    # Primeiro, localize o elemento com o XPath
    elemento = driver.find_element(By.XPATH, "/html/body/div[1]/div[1]/div[2]/div/a[3]")

    # Depois, mova o mouse até esse elemento
    action = ActionChains(driver)
    action.move_to_element(elemento).perform()

    

    time.sleep(3)

    # Localiza o botão completo
    botao = driver.find_element(By.ID, "splitbutton-1041-btnWrap")

    # Move o mouse com deslocamento e clica (ex: 30px à direita, 5px abaixo do topo)
    ActionChains(driver).move_to_element_with_offset(botao, 5, 5).click().perform()

    # csv
    time.sleep(3)

    # Pressiona TAB 7 vezes
    for i in range(7):
        pyautogui.press('tab')
    pyautogui.press('enter')

    ReNomeArquivo(Nome_Arquivo)


# puxa funções
Loga_extrai('USER.1', 'SENHA', 'AA') # USER - SENHA - TAG
Loga_extrai('USER.2', 'SENHA', 'LVL2')
Loga_extrai('USER3@EMPRESA', 'SENHA', 'LVL')
Loga_extrai('USER4@EMPRESA', 'SENHA', 'SIS')

# Fecha site
driver.close()

# Limpa dados ------------------------------------------------------------------------------------------------

# Filtra apenas arquivos no padrão sitrack(...).csv
arquivos = [f for f in os.listdir(Path_download) if f.startswith('sitrack(') and f.endswith('.csv')]

# Lista para armazenar os DataFrames
lista_df = []

# Carrega cada CSV e anexa na lista
for arquivo in arquivos:
    caminho = os.path.join(Path_download, arquivo)
    df = pd.read_csv(caminho, encoding='utf-8', sep=';')
    df['Arquivo'] = arquivo  # adiciona uma coluna indicando a origem
    lista_df.append(df)

# Mescla todos os DataFrames - planilhas sistemas
df_final = pd.concat(lista_df, ignore_index=True)



# filtra DF
df_final = df_final.loc[:, ~df_final.columns.str.contains("^Unnamed")]
df_final = df_final.loc[:, ~df_final.columns.str.contains("Odômetro")]
df_final = df_final.loc[:, ~df_final.columns.str.contains("Altitude")]
df_final = df_final.loc[:, ~df_final.columns.str.contains("Validade")]
df_final = df_final.loc[:, ~df_final.columns.str.contains("Canal")]
df_final = df_final.loc[:, ~df_final.columns.str.contains("Direção")]
df_final = df_final.loc[:, ~df_final.columns.str.contains("Velocidade")]
df_final = df_final.loc[:, ~df_final.columns.str.contains("Evento")]
df_final['Dados'] = df_final['Dados'].str.replace(r'^.*?RST \| ', '', regex=True)

df_final[['VELOCIDADE', 'LIMITE', 'TEMPO']] = df_final['Dados'].str.extract(
    r'tempo:(?P<TEMPO>[\d,]+)\s*\|\s*max:(?P<VELOCIDADE>[\d,]+)\s*\|\s*limite:(?P<LIMITE>[\d,]+)'
)[['VELOCIDADE', 'LIMITE', 'TEMPO']]

for col in ['VELOCIDADE', 'LIMITE', 'TEMPO']:
    df_final[col] = df_final[col].str.replace(',', '.').astype(float)

# Move a coluna 'Localização' para antes da coluna 'Nome'
colunas = df_final.columns.tolist()
colunas.remove('Localização')
indice_nome = colunas.index('Nome')
colunas.insert(indice_nome, 'Localização')

# Reorganiza o DataFrame
df_final = df_final[colunas]

df_final['Localização'] = df_final['Localização'].str.replace(r', Minas.*', '', regex=True)
df_final['Motorista'] = df_final['Motorista'].str.replace(r'^.*?_', '', regex=True)
df_final['Motorista'] = df_final['Motorista'].str.replace(r'^VSR\s+', '', regex=True)
df_final['Motorista'] = df_final['Motorista'].str.replace(r'\sCPF.*', '', regex=True)
df_final['Motorista'] = df_final['Motorista'].str.replace(r'\sDNI:.*', '', regex=True)

df_final.drop(columns=['Dados'], inplace=True)
df_final['Grupo'] = df_final['Arquivo'].str.extract(r'\((.*?)\)')

# Alfabética A-Z
df_final.sort_values(by='Motorista', ascending=True, inplace=True)

# coluna com %
df_final["% De excesso"] = ((df_final["VELOCIDADE"] - df_final["LIMITE"]) / df_final["LIMITE"]) * 100
df_final["% De excesso"] = df_final["% De excesso"].apply(lambda x: f"{round(x)}%" if x > 0 else "0%")

# Tira nomes vazios
df_final = df_final[df_final["Motorista"].notna() & (df_final["Motorista"].str.strip() != "")]

# Muda Nome por VSR
df_final.rename(columns={"Nome": "VSR"}, inplace=True)

# Apaga coluna Arquivo
df_final.drop(columns=["Arquivo"], inplace=True)

# importa e junta planilhas
# Caminho da planilha com nomes e unidades
arquivo_unidades = r"C:\Users\mauricio.vargas\Documents\Projetos\05 - EMPRESAX\SitrackMot.xlsx"

# Lê a planilha com nomes e unidades
df_unidades = pd.read_excel(arquivo_unidades)


# Faz merge com base na coluna de nome
df_final = df_final.merge(df_unidades, on="Motorista", how="left")

# Apaga colunas
df_final.drop(columns=["Cliente", "Zona próxima", "PosiçãoId"], inplace=True)

# Cria resumo
# Agrupa por motorista e grupo, conta o número de ocorrências
df_resumo = df_final.groupby(["Motorista", "Grupo","UNIDADE"]).size().reset_index(name="Ocorrências")

# Ordena pelo número de ocorrências (opcional)
df_resumo = df_resumo.sort_values(by="Ocorrências", ascending=False).reset_index(drop=True)

# coluna com Período
df_resumo["Período"] = f"{data_inicio} à {data_final}"

# ----------------------------------------------------------------------------------------------------
# 1 ------------------------------------------------------------------------
# Gera gráfico de pizza - Ocorrências por Unidade com legenda
fig1, ax1 = plt.subplots(figsize=(6, 6))

# Agrupa dados
dados = df_resumo.groupby("UNIDADE")["Ocorrências"].sum()

# Cria o gráfico de pizza sem rótulos diretos
wedges, texts = ax1.pie(
    dados,
    startangle=90,
    textprops={'fontsize': 9},
    labels=None,     # Remove rótulos das fatias
    autopct=None     # Remove % diretas
)

# Cria a legenda com % calculadas
total = dados.sum()
legendas = [f"{unidade} - {valor / total:.1%}" for unidade, valor in dados.items()]
ax1.legend(wedges, legendas, title="Unidades", loc="center left", bbox_to_anchor=(1, 0.5), fontsize=9)

# Título
ax1.set_title("Ocorrências por Unidade", fontsize=11)

# Salva como imagem
buffer1 = BytesIO()
plt.savefig(buffer1, format='png', bbox_inches='tight')
buffer1.seek(0)
img1 = base64.b64encode(buffer1.read()).decode()
plt.close()

# 2 ------------------------------------------------------------------------
# Gera gráfico de barras - Top 5 motoristas
fig2, ax2 = plt.subplots()
df_resumo.sort_values("Ocorrências", ascending=False).head(5).set_index("Motorista")["Ocorrências"].plot.barh(
    color="skyblue", ax=ax2
)
ax2.set_title("Motoristas com Mais Ocorrências no período")
ax2.set_xlabel("Ocorrências")
plt.gca().invert_yaxis()

# Rótulos nas barras com lógica de posição
for bar in ax2.patches:
    width = bar.get_width()
    # Se o valor for pequeno, escreve fora da barra à direita
    if width < 20:
        ax2.text(width + 5, bar.get_y() + bar.get_height() / 2,
                 f"{int(width)}", va='center', ha='left', fontsize=10, color='black', fontweight='bold')
    else:
        ax2.text(width - 10, bar.get_y() + bar.get_height() / 2,
                 f"{int(width)}", va='center', ha='right', fontsize=10, color='black', fontweight='bold')


buffer2 = BytesIO()
plt.savefig(buffer2, format='png', bbox_inches='tight')
buffer2.seek(0)

img2 = base64.b64encode(buffer2.read()).decode()
plt.close()

# 3 ------------------------------------------------------------------------
# Gera gráfico de barras - Ocorrências por Unidade (números fora da barra se forem pequenos)
fig3, ax3 = plt.subplots(figsize=(7, 4))  # Aumenta largura para mais espaço

# Dados agrupados e ordenados
dados_unidade = df_resumo.groupby("UNIDADE")["Ocorrências"].sum().sort_values()

# Gráfico de barras horizontais
bars = dados_unidade.plot.barh(color="skyblue", ax=ax3)

# Título e rótulo do eixo
ax3.set_title("Ocorrências por Unidade", fontsize=11)
ax3.set_xlabel("Ocorrências")

# Rótulos nas barras com lógica de posição
for bar in bars.patches:
    width = bar.get_width()
    # Se o valor for pequeno, escreve fora da barra à direita
    if width < 20:
        ax3.text(width + 5, bar.get_y() + bar.get_height() / 2,
                 f"{int(width)}", va='center', ha='left', fontsize=10, color='black', fontweight='bold')
    else:
        ax3.text(width - 10, bar.get_y() + bar.get_height() / 2,
                 f"{int(width)}", va='center', ha='right', fontsize=10, color='black', fontweight='bold')

# Salvar imagem em buffer
buffer3 = BytesIO()
plt.savefig(buffer3, format='png', bbox_inches='tight')
buffer3.seek(0)
img3 = base64.b64encode(buffer3.read()).decode()
plt.close()

# ----------------------------------------------------------------------------------------------------
# Mapa dinâmico

# Substitui vírgula por ponto e converte para float
df_final["Latitude"] = df_final["Latitude"].astype(str).str.replace(",", ".").astype(float)
df_final["Longitude"] = df_final["Longitude"].astype(str).str.replace(",", ".").astype(float)


fig = px.scatter_mapbox(
    df_final,
    lat="Latitude",
    lon="Longitude",
    hover_name="Localização",  # Exibe a localização ao passar o mouse
    hover_data=["Placa", "VELOCIDADE", "LIMITE", "% De excesso"],
    zoom=10,
    height=600,
    mapbox_style="open-street-map"  # mapa bonito e gratuito
)

#fig.show()
Mapa_HTML = r"C:\Users\mauricio.vargas\Downloads\mapa_interativo.html"

fig.write_html(Mapa_HTML)




# envia email ------------------------------------------------------------------------------------------------

# Formata data em string
data_inicio_str = data_inicio.replace("/", "-")
data_final_str = data_final.replace("/", "-")

# Caminho de saída
caminho_saida = os.path.join(Path_download, f"Relatório de Desempenho  {data_inicio_str} à {data_final_str}.xlsx")
# Salva em Excel
df_final.to_excel(caminho_saida, index=False)

print(f"✅ Arquivo Excel salvo em: {caminho_saida}")


# formatando em inteiro
colunas_para_inteiro = ["VELOCIDADE", "LIMITE", "TEMPO"]
df_final[colunas_para_inteiro] = df_final[colunas_para_inteiro].astype(int)

# Convertendo o DataFrame estilizado para HTML sem índice
df_html_final = df_final.style.hide(axis="index").to_html()
df_html_resumo = df_resumo.style.hide(axis="index").to_html()

# Construir o conteúdo do e-mail com o DataFrame e o gráfico
conteudo_email = f"""
<html>
<head>
    <style>
        .valor {{
            color: green;
            font-weight: bold;
            font-size: 1.2em; 
        }}
        table {{
            font-family: Arial, sans-serif;
            border-collapse: collapse;
            width: 100%;
        }}
        table, th, td {{
            border: 1px solid black;
            padding: 8px;
        }}
    </style>
</head>
<body>
    <p>Relatório de desempenho diário <span style="color: #0070C0;">SITRACK</span> de {data_inicio} a {data_final} </p>
    <p>Em anexo segue planilha com relatório estendido</p>
    
    <br><br>
    
    <h2>Distribuição de Ocorrências por Unidade</h2>
    <img src="data:image/png;base64,{img1}" alt="Gráfico Unidade" width="450"/>

    <br><br>

    <h2>Top 5 Motoristas com Mais Ocorrências</h2>
    <img src="data:image/png;base64,{img2}" alt="Gráfico Top 5 Motoristas" width="450"/>

    <br><br>

    <h2>Ocorrências por Unidade</h2>
    <img src="data:image/png;base64,{img3}" alt="Ocorrências por Unidade" width="450"/>

    <br>
    
    <h1>Ocorrências de Motoristas Resumo</h1>
    {df_html_resumo}  <!-- Inclui o DataFrame em formato HTML -->

    <br>

    <p>Atenciosamente,</p>
    <p>Mauricio Vargas</p>
</body>
</html>
"""

# Configuração do e-mail
if Valida_email_teste == '1':
    destinatario = "TESTE@teste1.com.br"
else:
    destinatario = (
        "supervisor de filiais@EMPRESAX.com.br;"
        "garagemEMPRESAX@EMPRESAX.com.br;"
        "monitoramento@EMPRESAX.com.br;"
        "gerencia@EMPRESAX.com.br;"
        "segurança@EMPRESAX.com.br;"
        "trafego@EMPRESAX.com.br;"
    )    
   
assunto = f"Relatório de desempenho Parcial - {data_inicio} À {data_final}"

corpo = conteudo_email

# Iniciar o Outlook
outlook = win32.Dispatch('Outlook.Application')

# Criar o e-mail
email = outlook.CreateItem(0)
email.To = destinatario
email.Subject = assunto
email.HTMLBody = corpo

email.Attachments.Add(Source=caminho_saida)
email.Attachments.Add(Source=Mapa_HTML)

# Enviar o e-mail
if Valida_email_teste == "2":
    pass
else:
    email.Send()

print("---------------------------------------------------------------------")
print(f"E-mail enviado com sucesso!")
# -------------------------------------------------------------------------------------------------------------------------------------------



# Apaga arquivos sitrack.csv
if Apg_arq_temp == "1":
    print("Nenhum arquivo apagado.")    
else:
    for file_name in os.listdir(r'C:\Users\mauricio.vargas\Downloads'):
        if "sitrack" in file_name:
            file_path = os.path.join(r'C:\Users\mauricio.vargas\Downloads', file_name)
            os.remove(file_path)
            print(f"Arquivo apagado: {file_path}")
        else:
            #print("Nenhum arquivo 'sitrack' encontrado na pasta Downloads.")
            pass
            
if Apg_arq_temp == "1":    
    pass
else:
    os.remove(caminho_saida)



## OBSOLETO EMPRESA Z MONITORAMENTO ↓

In [ ]:
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import win32com.client as win32
from selenium import webdriver
import matplotlib.pyplot as plt
from datetime import datetime
from io import BytesIO
import pandas as pd
import pyautogui
import base64
import time
import os



# Config data 
data_inicio = input('Insira a data de início (dd/mm/aaaa): ')
data_final = input('Insira a data final (dd/mm/aaaa): ')
Valida_email_teste = input("Digite 1 para teste, 2 para nao enviar, ou qualquer botão para enviar aos colaboradores.")
Apg_arq_temp = input("Aperte 1 para manter arquivos temporários.")

# Apaga arquivos sitrack.csv
for file_name in os.listdir(r'C:\Users\mauricio.vargas\Downloads'):
    if "sitrack" in file_name:
        file_path = os.path.join(r'C:\Users\mauricio.vargas\Downloads', file_name)
        os.remove(file_path)
        print(f"Arquivo apagado: {file_path}")
        
else:
    #print("Nenhum arquivo com 'sitrack.csv' encontrado na pasta Downloads.")
    pass

# Caminho Download
Path_download = r'C:\Users\mauricio.vargas\Downloads'

# Inicializa driver com ChromeDriverManager
options = webdriver.ChromeOptions()
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
driver.maximize_window()


def ReNomeArquivo(Nome_Arquivo):

    while True:
        encontrado = False
        for file_name in os.listdir(Path_download):
            if "sitrackPR" in file_name:
                encontrado = True
                old_path = os.path.join(Path_download, file_name)
                Nome_Arquivo = f'sitrack({Nome_Arquivo}).csv'
                new_path = os.path.join(Path_download, Nome_Arquivo)

                os.rename(old_path, new_path)
                print(f"Arquivo renomeado: {Nome_Arquivo}")
                return  # encerra a função após sucesso

        if not encontrado:
            print("Aguardando arquivo com 'sitrackPR' na pasta Downloads...")
            time.sleep(2)

def Loga_extrai (usuario_AA, senha_AA, Nome_Arquivo):
    url_login = 'https://www.URL.com.br/login DO SISTEMA/'
    driver.get(url_login)


    driver.find_element(By.NAME, "userName").clear()
    driver.find_element(By.NAME, "userName").send_keys(usuario_AA) 

    driver.find_element(By.NAME, "password").clear()
    driver.find_element(By.NAME, "password").send_keys(senha_AA) 

    driver.find_element(By.ID, "loginButton").click()


    driver.find_element(By.CSS_SELECTOR, "[title='Consultas']").click()

    time.sleep(0.5)

    driver.find_element(By.CSS_SELECTOR, "[title='Históricos']").click()

    time.sleep(0.5)

    driver.find_element(By.CSS_SELECTOR, "[title='H. Posição']").click()

    # - acessa o IFRAME
    iframe = driver.find_element(By.NAME, "tab3")  # Substitua "tab3" pelo atributo adequado do iframe, se necessário
    driver.switch_to.frame(iframe)

    time.sleep(0.5)

    driver.find_element(By.NAME, "rawdatefield-1037-inputEl").clear()
    driver.find_element(By.NAME, "rawdatefield-1037-inputEl").send_keys(data_inicio) 

    driver.find_element(By.NAME, "rawdatefield-1039-inputEl").clear()
    driver.find_element(By.NAME, "rawdatefield-1039-inputEl").send_keys(data_final) 

    driver.find_element(By.ID, "splitbutton-1041").click()
    # ----------------------------------------------------------------------------------------------------------------
    
    Tabela_Evento = None
    trigger = None
    action = None

    # Localiza o título da coluna com texto "Evento"
    Tabela_Evento = WebDriverWait(driver, 3).until(
        EC.visibility_of_element_located((By.XPATH, "//span[text()='Evento']/ancestor::div[contains(@class, 'x-column-header')]"))
    )
    ActionChains(driver).move_to_element(Tabela_Evento).perform()

    # Aguarda a seta (trigger) que aparece no hover
    trigger = WebDriverWait(driver, 3).until(
        EC.element_to_be_clickable((By.XPATH, "//span[text()='Evento']/ancestor::div[contains(@class, 'x-column-header')]//div[contains(@id, 'triggerEl')]"))
    )
    driver.execute_script("arguments[0].scrollIntoView(true);", trigger)
    trigger.click()

    # ----------------------------------------------------------------------------------------------------------------
    time.sleep(3)
    
    WebDriverWait(driver, 3).until(EC.element_to_be_clickable((By.XPATH, "//span[text()='Colunas']"))).click()

    # motorista 
    # Espera o elemento aparecer
    elemento = WebDriverWait(driver, 3).until(
        EC.presence_of_element_located((By.XPATH, "/html/body/div[7]/div/div[2]/div[2]/div[21]"))
    )

    # Rola até ele ficar visível
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", elemento)

    # Espera estar clicável e clica
    WebDriverWait(driver, 3).until(
        EC.element_to_be_clickable((By.XPATH, "/html/body/div[7]/div/div[2]/div[2]/div[21]"))
    ).click()

    # ----------------------------------------------------------------------------------------------------------------
    WebDriverWait(driver, 3).until(EC.element_to_be_clickable((By.XPATH, "//span[text()='Condições']"))).click()   
    
    time.sleep(1)

    pyautogui.press('tab')
    pyautogui.press('tab')

    pyautogui.sleep(2)

    # Escreve o texto
    pyautogui.write('em velocidade')
    time.sleep(2)   
    pyautogui.press('down')
    time.sleep(1)
    pyautogui.press('enter')
    pyautogui.press('tab')
    pyautogui.press('tab')
    pyautogui.press('enter')
    # --------------------------------------------------------------------

    time.sleep(3)

    # Primeiro, localize o elemento com o XPath
    elemento = driver.find_element(By.XPATH, "/html/body/div[1]/div[1]/div[2]/div/a[3]")

    # Depois, mova o mouse até esse elemento
    action = ActionChains(driver)
    action.move_to_element(elemento).perform()

    time.sleep(3)

    # Localiza o botão completo
    botao = driver.find_element(By.ID, "splitbutton-1041-btnWrap")

    # Move o mouse com deslocamento e clica (ex: 30px à direita, 5px abaixo do topo)
    ActionChains(driver).move_to_element_with_offset(botao, 5, 5).click().perform()

    # csv
    time.sleep(3)

    # Pressiona TAB 7 vezes
    for i in range(7):
        pyautogui.press('tab')
    pyautogui.press('enter')

    ReNomeArquivo(Nome_Arquivo)


# puxa funções
Loga_extrai('USER.1', 'SENHA', 'AA') # USER - SENHA - TAG
Loga_extrai('USER.2', 'SENHA', 'LVL2')
Loga_extrai('USER3@EMPRESA', 'SENHA', 'LVL')
Loga_extrai('USER4@EMPRESA', 'SENHA', 'SIS')

# Fecha site
driver.close()

# Limpa dados ------------------------------------------------------------------------------------------------

# Filtra apenas arquivos no padrão sitrack(...).csv
arquivos = [f for f in os.listdir(Path_download) if f.startswith('sitrack(') and f.endswith('.csv')]

# Lista para armazenar os DataFrames
lista_df = []

# Carrega cada CSV e anexa na lista
for arquivo in arquivos:
    caminho = os.path.join(Path_download, arquivo)
    df = pd.read_csv(caminho, encoding='utf-8', sep=';')
    df['Arquivo'] = arquivo  # adiciona uma coluna indicando a origem
    lista_df.append(df)

# Mescla todos os DataFrames - planilhas sistemas
df_final = pd.concat(lista_df, ignore_index=True)



# filtra DF
df_final = df_final.loc[:, ~df_final.columns.str.contains("^Unnamed")]
df_final = df_final.loc[:, ~df_final.columns.str.contains("Odômetro")]
df_final = df_final.loc[:, ~df_final.columns.str.contains("Altitude")]
df_final = df_final.loc[:, ~df_final.columns.str.contains("Validade")]
df_final = df_final.loc[:, ~df_final.columns.str.contains("Canal")]
df_final = df_final.loc[:, ~df_final.columns.str.contains("Direção")]
df_final = df_final.loc[:, ~df_final.columns.str.contains("Velocidade")]
df_final = df_final.loc[:, ~df_final.columns.str.contains("Evento")]
df_final['Dados'] = df_final['Dados'].str.replace(r'^.*?RST \| ', '', regex=True)

df_final[['VELOCIDADE', 'LIMITE', 'TEMPO']] = df_final['Dados'].str.extract(
    r'tempo:(?P<TEMPO>[\d,]+)\s*\|\s*max:(?P<VELOCIDADE>[\d,]+)\s*\|\s*limite:(?P<LIMITE>[\d,]+)'
)[['VELOCIDADE', 'LIMITE', 'TEMPO']]

for col in ['VELOCIDADE', 'LIMITE', 'TEMPO']:
    df_final[col] = df_final[col].str.replace(',', '.').astype(float)

# Move a coluna 'Localização' para antes da coluna 'Nome'
colunas = df_final.columns.tolist()
colunas.remove('Localização')
indice_nome = colunas.index('Nome')
colunas.insert(indice_nome, 'Localização')

# Reorganiza o DataFrame
df_final = df_final[colunas]

df_final['Localização'] = df_final['Localização'].str.replace(r', Minas.*', '', regex=True)
df_final['Motorista'] = df_final['Motorista'].str.replace(r'^.*?_', '', regex=True)
df_final['Motorista'] = df_final['Motorista'].str.replace(r'^VSR\s+', '', regex=True)
df_final['Motorista'] = df_final['Motorista'].str.replace(r'\sCPF.*', '', regex=True)
df_final['Motorista'] = df_final['Motorista'].str.replace(r'\sDNI:.*', '', regex=True)

df_final.drop(columns=['Dados'], inplace=True)
df_final['Grupo'] = df_final['Arquivo'].str.extract(r'\((.*?)\)')

# Alfabética A-Z
df_final.sort_values(by='Motorista', ascending=True, inplace=True)

# coluna com %
df_final["% De excesso"] = ((df_final["VELOCIDADE"] - df_final["LIMITE"]) / df_final["LIMITE"]) * 100
df_final["% De excesso"] = df_final["% De excesso"].apply(lambda x: f"{round(x)}%" if x > 0 else "0%")

# Tira nomes vazios
df_final = df_final[df_final["Motorista"].notna() & (df_final["Motorista"].str.strip() != "")]

# Muda Nome por VSR
df_final.rename(columns={"Nome": "VSR"}, inplace=True)

# Apaga coluna Arquivo
df_final.drop(columns=["Arquivo"], inplace=True)

# importa e junta planilhas
# Caminho da planilha com nomes e unidades
arquivo_unidades = r"C:\Users\mauricio.vargas\Documents\Projetos\05 - EMPRESAX\SitrackMot.xlsx"

# Lê a planilha com nomes e unidades
df_unidades = pd.read_excel(arquivo_unidades)


# Faz merge com base na coluna de nome
df_final = df_final.merge(df_unidades, on="Motorista", how="left")

# Cria resumo
# Agrupa por motorista e grupo, conta o número de ocorrências
df_resumo = df_final.groupby(["Motorista", "Grupo","UNIDADE"]).size().reset_index(name="Ocorrências")

# Ordena pelo número de ocorrências (opcional)
df_resumo = df_resumo.sort_values(by="Ocorrências", ascending=False).reset_index(drop=True)

# coluna com Período
df_resumo["Período"] = f"{data_inicio} à {data_final}"

# ----------------------------------------------------------------------------------------------------
# 1 ------------------------------------------------------------------------
# Gera gráfico de pizza - Ocorrências por Unidade com legenda
fig1, ax1 = plt.subplots(figsize=(6, 6))

# Agrupa dados
dados = df_resumo.groupby("UNIDADE")["Ocorrências"].sum()

# Cria o gráfico de pizza sem rótulos diretos
wedges, texts = ax1.pie(
    dados,
    startangle=90,
    textprops={'fontsize': 9},
    labels=None,     # Remove rótulos das fatias
    autopct=None     # Remove % diretas
)

# Cria a legenda com % calculadas
total = dados.sum()
legendas = [f"{unidade} - {valor / total:.1%}" for unidade, valor in dados.items()]
ax1.legend(wedges, legendas, title="Unidades", loc="center left", bbox_to_anchor=(1, 0.5), fontsize=9)

# Título
ax1.set_title("Ocorrências por Unidade", fontsize=11)

# Salva como imagem
buffer1 = BytesIO()
plt.savefig(buffer1, format='png', bbox_inches='tight')
buffer1.seek(0)
img1 = base64.b64encode(buffer1.read()).decode()
plt.close()

# 2 ------------------------------------------------------------------------
# Gera gráfico de barras - Top 5 motoristas
fig2, ax2 = plt.subplots()
df_resumo.sort_values("Ocorrências", ascending=False).head(5).set_index("Motorista")["Ocorrências"].plot.barh(
    color="skyblue", ax=ax2
)
ax2.set_title("Motoristas com Mais Ocorrências no período")
ax2.set_xlabel("Ocorrências")
plt.gca().invert_yaxis()

# Rótulos nas barras com lógica de posição
for bar in ax2.patches:
    width = bar.get_width()
    # Se o valor for pequeno, escreve fora da barra à direita
    if width < 20:
        ax2.text(width + 5, bar.get_y() + bar.get_height() / 2,
                 f"{int(width)}", va='center', ha='left', fontsize=10, color='black', fontweight='bold')
    else:
        ax2.text(width - 10, bar.get_y() + bar.get_height() / 2,
                 f"{int(width)}", va='center', ha='right', fontsize=10, color='black', fontweight='bold')


buffer2 = BytesIO()
plt.savefig(buffer2, format='png', bbox_inches='tight')
buffer2.seek(0)

img2 = base64.b64encode(buffer2.read()).decode()
plt.close()

# 3 ------------------------------------------------------------------------
# Gera gráfico de barras - Ocorrências por Unidade (números fora da barra se forem pequenos)
fig3, ax3 = plt.subplots(figsize=(7, 4))  # Aumenta largura para mais espaço

# Dados agrupados e ordenados
dados_unidade = df_resumo.groupby("UNIDADE")["Ocorrências"].sum().sort_values()

# Gráfico de barras horizontais
bars = dados_unidade.plot.barh(color="skyblue", ax=ax3)

# Título e rótulo do eixo
ax3.set_title("Ocorrências por Unidade", fontsize=11)
ax3.set_xlabel("Ocorrências")

# Rótulos nas barras com lógica de posição
for bar in bars.patches:
    width = bar.get_width()
    # Se o valor for pequeno, escreve fora da barra à direita
    if width < 20:
        ax3.text(width + 5, bar.get_y() + bar.get_height() / 2,
                 f"{int(width)}", va='center', ha='left', fontsize=10, color='black', fontweight='bold')
    else:
        ax3.text(width - 10, bar.get_y() + bar.get_height() / 2,
                 f"{int(width)}", va='center', ha='right', fontsize=10, color='black', fontweight='bold')

# Salvar imagem em buffer
buffer3 = BytesIO()
plt.savefig(buffer3, format='png', bbox_inches='tight')
buffer3.seek(0)
img3 = base64.b64encode(buffer3.read()).decode()
plt.close()

# ----------------------------------------------------------------------------------------------------

# envia email ------------------------------------------------------------------------------------------------

# Formata data em string
data_inicio_str = data_inicio.replace("/", "-")
data_final_str = data_final.replace("/", "-")

# Caminho de saída
caminho_saida = os.path.join(Path_download, f"Relatório de Desempenho  {data_inicio_str} à {data_final_str}.xlsx")
# Salva em Excel
df_final.to_excel(caminho_saida, index=False)

print(f"✅ Arquivo Excel salvo em: {caminho_saida}")


# formatando em inteiro
colunas_para_inteiro = ["VELOCIDADE", "LIMITE", "TEMPO"]
df_final[colunas_para_inteiro] = df_final[colunas_para_inteiro].astype(int)

# Convertendo o DataFrame estilizado para HTML sem índice
df_html_final = df_final.style.hide(axis="index").to_html()
df_html_resumo = df_resumo.style.hide(axis="index").to_html()

# Construir o conteúdo do e-mail com o DataFrame e o gráfico
conteudo_email = f"""
<html>
<head>
    <style>
        .valor {{
            color: green;
            font-weight: bold;
            font-size: 1.2em; 
        }}
        table {{
            font-family: Arial, sans-serif;
            border-collapse: collapse;
            width: 100%;
        }}
        table, th, td {{
            border: 1px solid black;
            padding: 8px;
        }}
    </style>
</head>
<body>
    <p>Relatório de desempenho diário <span style="color: #0070C0;">SITRACK</span> de {data_inicio} a {data_final} </p>
    <p>Em anexo segue planilha com relatório estendido</p>
    
    <br><br>
    
    <h2>Distribuição de Ocorrências por Unidade</h2>
    <img src="data:image/png;base64,{img1}" alt="Gráfico Unidade" width="450"/>

    <br><br>

    <h2>Top 5 Motoristas com Mais Ocorrências</h2>
    <img src="data:image/png;base64,{img2}" alt="Gráfico Top 5 Motoristas" width="450"/>

    <br><br>

    <h2>Ocorrências por Unidade</h2>
    <img src="data:image/png;base64,{img3}" alt="Ocorrências por Unidade" width="450"/>

    <br>
    
    <h1>Ocorrências de Motoristas Resumo</h1>
    {df_html_resumo}  <!-- Inclui o DataFrame em formato HTML -->

    <br>

    <p>Atenciosamente,</p>
    <p>Mauricio Vargas</p>
</body>
</html>
"""

# Configuração do e-mail
if Valida_email_teste == '1':
    destinatario = "TESTE@teste1.com.br"
else:
    destinatario = (
        "todos@EMPRESAX.com.br;"
    )    
   
assunto = f"Relatório de desempenho Parcial - {data_inicio} À {data_final}"

corpo = conteudo_email

# Iniciar o Outlook
outlook = win32.Dispatch('Outlook.Application')

# Criar o e-mail
email = outlook.CreateItem(0)
email.To = destinatario
email.Subject = assunto
email.HTMLBody = corpo

email.Attachments.Add(Source=caminho_saida)

# Enviar o e-mail
if Valida_email_teste == "2":
    pass
else:
    email.Send()

print("---------------------------------------------------------------------")
print(f"E-mail enviado com sucesso!")
# -------------------------------------------------------------------------------------------------------------------------------------------



# Apaga arquivos sitrack.csv
if Apg_arq_temp == "1":
    print("Nenhum arquivo apagado.")    
else:
    for file_name in os.listdir(r'C:\Users\mauricio.vargas\Downloads'):
        if "sitrack" in file_name:
            file_path = os.path.join(r'C:\Users\mauricio.vargas\Downloads', file_name)
            os.remove(file_path)
            print(f"Arquivo apagado: {file_path}")
        else:
            #print("Nenhum arquivo 'sitrack' encontrado na pasta Downloads.")
            pass
            
if Apg_arq_temp == "1":    
    pass
else:
    os.remove(caminho_saida)




## Enviar relatório EMPRESA Y ↓X
Execute para enviar o relatório via email

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time
import os
import xml.etree.ElementTree as ET
import pandas as pd
from datetime import datetime, timedelta
import zipfile
import win32com.client


DATA_INICIO = input("insira a data de inicio:")
DATA_FIM = input("insira a data de fim:")
Valida_email_teste = input("Digite 1 para teste, ou qualquer botão para enviar aos colaboradores.")
Valida_email_teste = '1'

# Caminho manual para o ChromeDriver
base_dir = os.path.expanduser(r'~\.wdm\drivers\chromedriver\win64\127.0.6533.72\chromedriver-win32')
chrome_driver_path = os.path.join(base_dir, 'chromedriver.exe')
service = Service(ChromeDriverManager().install())
options = webdriver.ChromeOptions()

# Inicializando o driver
driver = webdriver.Chrome(service=service, options=options)


# Abre o site de login
url_login = 'https://URL.com.br/admin/login' 
driver.get(url_login)

# Defina as informações de login
usuario = 'USER'
senha = 'SENHA'

# Encontra o campo de usuário e insere o nome de usuário
campo_usuario = driver.find_element(By.ID, "usuario") 
campo_usuario.clear()
campo_usuario.send_keys(usuario)
campo_senha = driver.find_element(By.ID, "senha")
campo_senha.clear()
campo_senha.send_keys(senha)

botao_login = driver.find_element(By.ID, "entrar")
botao_login.click()

time.sleep(5)

dropdown = driver.find_element(By.XPATH, "/html/body/aside/nav/div/ul/li[6]")
dropdown.click()


ExportaXML = driver.find_element(By.XPATH, "/html/body/aside/nav/div/ul/li[6]/ul/div[4]/div/div/li[11]")
ExportaXML.click()



# Insere data
time.sleep(3)
DATAINI = driver.find_element(By.XPATH, "/html/body/div[1]/div[1]/div[5]/div/div/div[2]/form/div/div/div[1]/div/div/input[1]")
DATAINI.clear()
DATAINI.send_keys(DATA_INICIO)

DATAF = driver.find_element(By.XPATH, "/html/body/div[1]/div[1]/div[5]/div/div/div[2]/form/div/div/div[1]/div/div/input[2]")
DATAF.clear()
DATAF.send_keys(DATA_FIM)


GeraXML = driver.find_element(By.XPATH, "/html/body/div[1]/div[1]/div[5]/div/div/div[3]/button[2]")
GeraXML.click()




# Processamento de dados XML -----------------------------------------------------------------------------------------------------------------
#---------------------------------------------------------------------------------------------------------------------------------------------
def encontrar_arquivos_xml(diretorio):
    caminho_xml_zip = None
    encontrou_arquivo = False
    mensagem_nao_encontrado_impressa = False
    while caminho_xml_zip is None:
        for caminho_pasta, subpastas, arquivos in os.walk(diretorio):
            for nome_arquivo in arquivos:
                if "XML" in nome_arquivo.upper():
                    print(f"Arquivo XML encontrado: {nome_arquivo}")
                    encontrou_arquivo = True
                    caminho_xml_zip = os.path.join(caminho_pasta, nome_arquivo)
                    break
            if caminho_xml_zip is not None:
                break
        else:
            if not mensagem_nao_encontrado_impressa:
                print("Arquivo XML nãbpao encontrado. Tentando novamente em 5 segundos...")
                mensagem_nao_encontrado_impressa = True
            time.sleep(5)
    return caminho_xml_zip

#---------------------------------------------------------------------------------------------------------------------------------------------

def extrair_arquivos_xml(caminho_zip, diretorio_destino):    
    caminhos_xml = []

    # Abre o arquivo ZIP
    with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:
        # Extrai todos os arquivos
        zip_ref.extractall(diretorio_destino)
        
        # Percorre os arquivos extraídos
        for nome_arquivo in zip_ref.namelist():
            if "XML" in nome_arquivo.upper():
                caminho_xml = diretorio_destino + "/" + nome_arquivo
                caminhos_xml.append(caminho_xml)
                #print(f"Arquivo XML extraído e organizado: {caminho_xml}")
    return caminhos_xml

#---------------------------------------------------------------------------------------------------------------------------------------------

def processar_arquivos_xml(caminhos_xml, elementos_desejados):
    dados = []
    namespace = {'ns': 'http://www.portalfiscal.inf.br/bpe'} 
    
    for caminho_xml in caminhos_xml:
        try:
            tree = ET.parse(caminho_xml)
            root = tree.getroot()
            #print("-------------------------------------------------------------------------------------------------------------------------------")
            #print(f"Conteúdo do arquivo XML: {caminho_xml}")
            
            # Dicionário para armazenar os dados de um único arquivo XML
            dados_arquivo = {}
            
            # Percorre e extrai os elementos desejados do XML
            for elem_tag in elementos_desejados:
                elem = root.find(f'.//ns:{elem_tag}', namespaces=namespace)
                if elem is not None:
                    dados_arquivo[elem.tag.split('}')[-1]] = elem.text
                    #print(f"{elem.tag.split('}')[-1]}: {elem.text}")
                    
            # Adiciona os dados do arquivo XML na lista de dados
            dados.append(dados_arquivo)
    
        except ET.ParseError as e:
            print(f"Erro ao analisar o arquivo {caminho_xml}: {e}")
        except FileNotFoundError as e:
            print(f"Arquivo não encontrado: {e}")
        except Exception as e:
            print(f"Ocorreu um erro ao processar o arquivo {caminho_xml}: {e}")
    df = pd.DataFrame(dados)        
    return df

#---------------------------------------------------------------------------------------------------------------------------------------------
def apagar_arquivos_xml(diretorio_destino, extencao):
    # Listar todos os arquivos no diretório de destino
    for nome_arquivo in os.listdir(diretorio_destino):
        # Verificar se o arquivo tem a extensão 
        if nome_arquivo.endswith(extencao):
            caminho_arquivo = os.path.join(diretorio_destino, nome_arquivo)
            try:
                # Apagar o arquivo
                os.remove(caminho_arquivo)
            except Exception as e:
                print(f"Erro ao apagar o arquivo {caminho_arquivo}: {e}")
                
#---------------------------------------------------------------------------------------------------------------------------------------------
# Função para enviar e-mail usando o Outlook
def enviar_email_outlook(destinatario, assunto, corpo):
    outlook = win32com.client.Dispatch('outlook.application')
    email = outlook.CreateItem(0)
    email.To = destinatario
    email.Subject = assunto
    email.HTMLBody = corpo
    email.Send()
    
#---------------------------------------------------------------------------------------------------------------------------------------------

# Caminhos e elementos a serem filtrados
#diretorio = "C:/Users/mauricio.vargas/Downloads"
#diretorio_destino = "C:/Users/mauricio.vargas/Documents/Projetos/Bpa_indicadores"
diretorio = os.path.join(os.path.expanduser("~"), "Downloads")
diretorio_destino = os.path.join(os.path.expanduser("~"), "Documentos")



caminho_xml_zip = encontrar_arquivos_xml(diretorio)
elementos_desejados = ["nBP", "vBP", "xLocOrig", "xLocDest"]

driver.quit()

# Extrair arquivos XML do ZIP e converter em DataFrame
caminhos_xml = extrair_arquivos_xml(caminho_xml_zip, diretorio_destino)
df = processar_arquivos_xml(caminhos_xml, elementos_desejados)

# Converter a coluna vBP para numérico
df['vBP'] = df['vBP'].str.replace(',', '.')
df['vBP'] = pd.to_numeric(df['vBP'])
soma_vbp = df['vBP'].sum()
num_bp = df['vBP'].count()

# Limpa arquivos Xml temporários
apagar_arquivos_xml(diretorio_destino, ".xml")
apagar_arquivos_xml(diretorio, ".zip")
# Exibir a soma
print("-------------------------------------------------------------------------------------------------------------------------------")


#data
DATA_INICIO = DATA_INICIO.replace('/', '-')
DATA_FIM = DATA_FIM.replace('/', '-')
data_inicio = datetime.strptime(DATA_INICIO, '%d-%m-%Y')
data_fim = datetime.strptime(DATA_FIM, '%d-%m-%Y')
N_Dias = (data_fim - data_inicio + timedelta(days=1)).days

#calculo de KM
N_Viagens_dia = 4
N_Viagens_dia = N_Viagens_dia * N_Dias
Km_Viagem = 227.9
Km_total_bpa = (Km_Viagem * N_Viagens_dia)

soma_vbp_str = f"{soma_vbp:,.2f}".replace('.', '*').replace(',', '.').replace('*', ',')
num_bp_str = f"{num_bp:,}".replace(',', '.')
Km_total_bpa_str = f"{Km_total_bpa:,.2f}".replace('.', '*').replace(',', '.').replace('*', ',')

RPK = soma_vbp / Km_total_bpa
RPK_str = f"{RPK:,.2f}".replace('.', ',')

print(f"Relatório parcial de vendas Bpa de {DATA_INICIO} a {DATA_FIM}")
print(f"O total faturado é: R${soma_vbp:,.2f}".replace('.', '*').replace(',', '.').replace('*', ','))
print(f"O total de bilhetes vendidos é: {num_bp:,}".replace(',', '.'))
print(f"Km total percorrido: {Km_total_bpa:,.2f}Km".replace('.', '*').replace(',', '.').replace('*', ','))
print(f"Receita por km: R${soma_vbp/Km_total_bpa:.2f}".replace('.', ','))

print("-------------------------------------------------------------------------------------------------------------------------------")

# Conteúdo do e-mail em HTML
conteudo_email = f"""
<html>
<head>
    <style>
        .valor {{
            color: green;
        }}
    </style>
</head>
<body>

    <p>Relatório parcial de vendas Bpa de {DATA_INICIO} À {DATA_FIM}</p>
    <p>O total faturado é: <span class="valor">R${soma_vbp_str}▲</span></p>
    <p>O total de bilhetes vendidos é: <span class="valor">{num_bp_str}</span></p>
    <p>Km total percorrido: <span class="valor">{Km_total_bpa_str}Km</span></p>
    <p>Receita por km: <span class="valor">R${RPK_str}▲</span></p>
    
    <br>
    <p>Atenciosamente,</p>
    <p>Mauricio Vargas</p>
</body>
</html>
"""



# Configuração do e-mail
if Valida_email_teste == '1':
    destinatario = "TESTE@teste1.com.br;"
else:
    destinatario = "EMPRESAX@EMPRESAX.com.br;r"


assunto = f"Relatório de Vendas Bpa - {DATA_INICIO} À {DATA_FIM}"
corpo = conteudo_email

# Enviar o e-mail
enviar_email_outlook(destinatario, assunto, corpo)

print("E-mail enviado com sucesso.")



## Extrair arquivos EMPRESAX ↓X
Execute para extrair arquivos

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.action_chains import ActionChains
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import matplotlib.pyplot as plt
import time
import pyautogui
import xlrd
from openpyxl import Workbook
from openpyxl.styles import Alignment
import openpyxl
from datetime import datetime, timedelta
import pandas as pd
import os
import win32com.client as win32
from io import BytesIO
import base64


# Config data 
data_inicio = input('Insira a data de início (dd/mm/aaaa): ')
data_final = input('Insira a data final (dd/mm/aaaa): ')



# Config caminhos 
desc_path = r'C:\Users\mauricio.vargas\Documents\Projetos\05 - EMPRESAX\DescServ.xlsx'


# Busca arquivo
for file_name in os.listdir(r'C:\Users\mauricio.vargas\Downloads'):
    if "RelatorioOrigemDestino" in file_name:
        file_path = os.path.join(r'C:\Users\mauricio.vargas\Downloads', file_name)
        os.remove(file_path)
        print(f"Arquivo apagado: {file_path}")
        
else:
    print("Nenhum arquivo com 'RelatorioOrigemDestino' encontrado na pasta Downloads.")


# -------------------------------------------------------------------------------------------------------------------------------------------
# Acessa Web service
service = Service(ChromeDriverManager().install())
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, timeout=60) 
driver.maximize_window()
url_login = 'https://URL.EMPRESAX'
driver.get(url_login)
usuario = 'USER'
senha = 'SENHA'

driver.find_element(By.NAME, "j_username").clear()
driver.find_element(By.NAME, "j_username").send_keys(usuario) 

driver.find_element(By.NAME, "j_password").clear()
driver.find_element(By.NAME, "j_password").send_keys(senha)

driver.find_element(By.NAME, "btnAcessar").click()



# -------------------------------------------------------------------------------------------------------------------------------------------
# Config data 
data_inicio_obj = datetime.strptime(data_inicio, "%d/%m/%Y")
data_final_obj = datetime.strptime(data_final, "%d/%m/%Y")

data_relatorio_obj = datetime.strptime(data_inicio, "%d/%m/%Y")
data_relatorio_str = data_relatorio_obj.strftime("%d/%m/%Y")


while data_relatorio_obj <= data_final_obj:
    print('Relatório dia:', data_relatorio_str)
    

    # -------------------------------------------------------------------------------------------------------------------------------------------
    # relatório caminho

    BotaoRelat = wait.until(EC.presence_of_element_located((By.XPATH, '/html/body/div/div/div[3]/div[2]/div[1]/div[1]/div/div/table/tbody/tr/td[9]')))
    BotaoRelat.click()

    time.sleep(1)

    element = driver.find_element(By.PARTIAL_LINK_TEXT, 'Relatórios')
    element.click()

    actions = ActionChains(driver)

    element = driver.find_element(By.PARTIAL_LINK_TEXT, 'Relatórios Estatísticos')
    actions.move_to_element(element).perform()

    element = driver.find_element(By.PARTIAL_LINK_TEXT, 'Desempenho Por Serviço')
    element.click()

    # config relatório ------------------------------------------------------------------------------------------------

    DataIni = wait.until(EC.presence_of_element_located((By.XPATH, '/html/body/div[2]/div[3]/div/div/div/div[1]/div[3]/table/tbody[2]/tr[1]/td[2]/div/i/input')))
    DataIni.click()
    DataIni.send_keys(data_relatorio_str) 
    DataFim = driver.find_element(By.XPATH, '/html/body/div[2]/div[3]/div/div/div/div[1]/div[3]/table/tbody[2]/tr[1]/td[4]/div/i/input')
    DataFim.click()
    DataFim.send_keys(data_relatorio_str) 

    Empresa = driver.find_element(By.XPATH, '/html/body/div[2]/div[3]/div/div/div/div[1]/div[3]/table/tbody[2]/tr[2]/td[2]/div/i/input')
    Empresa.click()
    time.sleep(1)
    Empresa = driver.find_element(By.XPATH, '/html/body/div[3]/table/tbody/tr/td[2]')
    Empresa.click()

    Botao = driver.find_element(By.XPATH, '/html/body/div[2]/div[3]/div/div/div/div[1]/div[3]/table/tbody[2]/tr[4]/td/div/span/input')
    Botao.click()

    Botao = driver.find_element(By.XPATH, '/html/body/div[2]/div[3]/div/div/div/div[1]/div[3]/table/tbody[2]/tr[5]/td/div/span/input')
    Botao.click()

    Botao = driver.find_element(By.XPATH, '/html/body/div[2]/div[3]/div/div/div/div[2]/div[1]/button')
    Botao.click()

    BotaoExcel = wait.until(EC.presence_of_element_located((By.XPATH, '/html/body/div[4]/div[3]/div/div/div/div/div[1]/div[1]/div[2]')))
    BotaoExcel.click()

    #pyautogui.click(1198,109,duration=3)

    time.sleep(3)
 

    # busca arquivo ------------------------------------------------------------------------------------------------
    for file_name in os.listdir(r'C:\Users\mauricio.vargas\Downloads'):
        if "RelatorioOrigemDestino" in file_name:
            file_path = os.path.join(r'C:\Users\mauricio.vargas\Downloads', file_name)
            print(f"Arquivo encontrado: {file_path}")
            break
    else:
        print("Nenhum arquivo com 'RelatorioOrigemDestino' encontrado na pasta Downloads.")
        
    # -------------------------------------------------------------------------------------------------------------------------------------------
    # busca arquivo

    # Abre o arquivo .xls com xlrd
    workbook = xlrd.open_workbook(file_path)
    sheet = workbook.sheet_by_name('RelatorioOrigemDestino')

    # Cria um novo arquivo Excel .xlsx para armazenar o filtro
    wb = Workbook()
    ws = wb.active
    ws.title = "Filtrado"

    # Copia o cabeçalho
    for col in range(sheet.ncols):
        ws.cell(row=1, column=col + 1, value=sheet.cell_value(0, col))

    # Filtra os dados (assumindo que 'Empresa' está na coluna 0) e remove linhas em branco
    row_index = 2
    for row in range(1, sheet.nrows):
        empresa = sheet.cell_value(row, 0)
        if empresa == 'EMPRESAX' or empresa == '':
            # Insere a linha completa, mesmo se houver células vazias
            for col in range(sheet.ncols):
                ws.cell(row=row_index, column=col + 1, value=sheet.cell_value(row, col))
            row_index += 1

    # Alinhar os dados centralizados
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
        for cell in row:
            cell.alignment = Alignment(horizontal="center", vertical="center")

    # Escreve "PASSAGEIROS" na célula G1 e "FATURAMENTO" na célula H1
    ws['G1'] = 'Passageiros'
    ws['H1'] = 'Faturamento'

    # Deleta as células de G2 a M2 e desloca os dados para cima
    for col in range(7, 14):
        for row in range(2, ws.max_row):
            ws.cell(row=row, column=col).value = ws.cell(row=row + 1, column=col).value

    # Limpa os valores duplicados na última linha
    for col in range(7, 14):
        ws.cell(row=ws.max_row, column=col).value = None

    # Deletar colunas específicas
    ws.delete_cols(1)  
    ws.delete_cols(2) 
    ws.delete_cols(2) 
    ws.delete_cols(2) 
    ws.delete_cols(2)  

    # Apaga os valores das colunas D até H
    for col in range(4, 9):
        for row in range(1, ws.max_row + 1):
            ws.cell(row=row, column=col).value = None

    # Aplicar filtro na primeira linha
    ws.auto_filter.ref = "A1:C24"

    # Ordenar a coluna "Serviço"
    data_rows = list(ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=3, values_only=True))
    sorted_rows = sorted([row for row in data_rows if row[0] is not None], key=lambda row: str(row[0]))

    # Limpa as linhas e reescreve com dados ordenados
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=3):
        for cell in row:
            cell.value = None

    for idx, row_data in enumerate(sorted_rows, start=2):
        for col_idx, value in enumerate(row_data, start=1):
            ws.cell(row=idx, column=col_idx, value=value)

    # Formatar "Faturamento" como valor monetário BRL
    for row in range(2, ws.max_row + 1):
        cell = ws.cell(row=row, column=3)
        if isinstance(cell.value, (int, float)):
            cell.number_format = '"R$"#,##0.00'

    data = []
    columns = [cell.value for cell in ws[1]] 

    # Percorrer as linhas restantes para coletar os dados
    for row in ws.iter_rows(min_row=2, values_only=True):
        if any(row): 
            data.append(dict(zip(columns, row)))

    # Transformar os dados extraídos em um DataFrame
    fin_df = pd.DataFrame(data)
    fin_df = fin_df.dropna(axis=1, how='all')

    # mescla arquivos ------------------------------------------------------------------------------------------------
    desc_df = pd.read_excel(desc_path)
    merged_df = pd.merge(fin_df, desc_df, on="Serviço", how="left")

    # ----------------------------------------------------------------------------------------------------------------
    # Preenche a coluna "Data" apenas para as linhas que têm dados nas colunas principais

    mask = merged_df[['Serviço', 'Passageiros', 'Faturamento']].notnull().all(axis=1)
    merged_df.loc[mask, 'Data'] = data_relatorio_str

    df_final = merged_df.dropna(subset=['Serviço', 'Passageiros', 'Faturamento'], how='any')


    df_final = df_final.copy()
    df_final['RPK'] = (df_final['Faturamento'] / df_final['KM Total da linha']).round(2)


    # Mover a nova coluna 'RPK' para logo após a coluna 'Meta' 
    cols = df_final.columns.tolist()
    meta_index = cols.index('Meta')
    cols.insert(meta_index + 1, cols.pop(cols.index('RPK')))
    df_final = df_final[cols]
    os.remove(file_path)

    # Agrupar os dados por 'Linha' e somar 'Faturamento' e 'KM Total da linha'
    df_resumo = df_final.groupby('Linha').agg({
        'Faturamento': 'sum',
        'KM Total da linha': 'sum',
    }).reset_index()

    # Adicionar a coluna 'Viagens', que conta o número de vezes que cada linha apareceu
    df_resumo['Viagens'] = df_final.groupby('Linha').size().values

    # Calcular o RPK como Faturamento Total / KM Total da Linha
    df_resumo['RPK'] = df_resumo['Faturamento'] / df_resumo['KM Total da linha']

    # Convertendo a coluna 'Linha' para string para garantir que cada linha seja tratada individualmente
    df_resumo['Linha'] = df_resumo['Linha'].astype(str)

    

    # -------------------------------------------------------------------------------------------------------------------------------------------
    # Salvando relatório em donwloads


    dia_relatorio = data_relatorio_str[:2]
    data_final_edit = data_final.replace('/', '-')
    df_final.to_excel(f"~/Downloads/RelServ {dia_relatorio} a {data_final_edit}.xlsx", index=False)
    print('Relatório salvo com sucesso na pasta "Downloads"!')

    # soma +1 na data do relatório
    data_relatorio_obj += timedelta(days=1)
    data_relatorio_str = data_relatorio_obj.strftime("%d/%m/%Y")
    driver.get(url_login)


# -------------------------------------------------------------------------------------------------------------------------------------------
# Concatena arquivos gerados na pasta downloads

diretorio_downloads = "C:/Users/mauricio.vargas/Downloads"

# Lista para armazenar cada DataFrame carregado
df_RelServ = []

# Percorrer todos os arquivos no diretório
for filename in os.listdir(diretorio_downloads):
    if "RelServ" in filename and filename.endswith(".xlsx"):
        file_path = os.path.join(diretorio_downloads, filename)
        df = pd.read_excel(file_path)  
        df_RelServ.append(df)  

# Concatenar todos os DataFrames em um único
df_RelServ_final = pd.concat(df_RelServ, ignore_index=True)

# Salvar o DataFrame final em um novo arquivo Excel
df_RelServ_final.to_excel("~/Downloads/Rel_concatenado.xlsx", index=False)



driver.quit()

# limpa todos os arquivos temporários
for filename in os.listdir(diretorio_downloads):
    if "RelServ" in filename and filename.endswith(".xlsx"):
        file_path  = os.path.join(diretorio_downloads, filename)
        os.remove(file_path)  # Usar os.remove() para deletar o arquivo
        print(f"Arquivo deletado: {file_path}")